In [1]:
import os
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv  
from langchain_core.messages import BaseMessage # The foundational class for all message types in LangGraph
from langchain_core.messages import SystemMessage # Message for providing instructions to the LLM
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode


load_dotenv()

google_api_key = os.environ["GOOGLE_API_KEY"]

free_model_names = [
    # Current stable Gemini models
    "gemini-3.8-flash",
    "gemini-3.7-flash",
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",

    # Preview models
    "gemini-3-flash-preview",

    # Legacy models - restricted/deprecated
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.5-pro",
]

models = [
    ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=google_api_key,
        max_retries=0,
        timeout=30,
    )
    for model_name in free_model_names
]
llm = models[0].with_fallbacks(models[1:])

# 🧠 Section 1: What are Guardrails?

Guardrails help you build safe, compliant AI applications by validating and filtering content at key points in your agent's execution.

They are implemented as middleware that intercepts execution:

Before the agent starts (input guardrails)

After it completes (output guardrails)

Around model and tool calls

Common Use Cases:

Use Case	                Example

PII leakage prevention	    Redact emails/credit cards before logging
Prompt injection blocking	Detect adversarial inputs
Harmful content filtering	Block dangerous requests
Business rule enforcement	Require approval for financial ops
Output quality validation	Ensure response meets safety standards

# ⚖️ Section 2: Two Approaches to Guardrails
Deterministic Guardrails

Rule-based: regex, keyword matching, explicit checks

✅ Fast, predictable, cost-effective

❌ May miss nuanced violations

# Model-Based Guardrails

Uses LLMs/classifiers for semantic understanding

✅ Catches subtle/nuanced issues

❌ Slower and more expensive

In [2]:
import re

# --- Deterministic approach ---
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


In [3]:
# --- Model-based approach ---
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = llm
    prompt = f"""Is the following user input safe to process? 
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    content = result.content
    if isinstance(content, list):
        content = next((b["text"] for b in content if isinstance(b, dict) and b.get("type") == "text"), str(content))
    return content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===
🚫 UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of France?
✅ SAFE: Explain how malware spreads


# 🔒 Section 3: Built-in Guardrail — PII Detection Middleware
LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

Supported PII Types:

Type	Example
email	user@example.com
credit_card	5105-1051-0510-5100
ip	192.168.1.1
mac_address	00:1A:2B:3C:4D:5E
url	https://secret-site.com

Strategies:
Strategy	Result
redact	[REDACTED_EMAIL]
mask	****-****-****-1234
hash	a8f5f167...
block	Raises an exception

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

# Create agent with PII Middleware
agent = create_agent(
    model=llm,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)
print("Agent with PII middleware created successfully.")

Agent with PII middleware created successfully.


In [5]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
[{'type': 'text', 'text': 'I found your customer record using the last four digits of your card. How can I help you today?', 'extras': {'signature': 'EnsKeQFpFH0TvXRYuWTJC934y4EQbOKSuk6ZgRNCat2WTdvB1gUVJ3oj6jPv+qW4icNm6wpmcVSwA+SoIelg5jM2Loqmsf3+l+uJgA/GTgarfB7vXYk7of+v+ScOdm+sz7mdBEPcc7BbfTOTP1kjrIElpQ3MkNa/MNcc6vU='}}]


In [6]:

# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


# 👤 Section 4: Built-in Guardrail — Human-in-the-Loop Middleware
Pauses agent execution before sensitive operations and waits for human approval.

Best for:

Financial transactions
Sending emails to external parties
Deleting production data
Any operation with significant business impact

Key requirement: A checkpointer for state persistence across interrupts.

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent = create_agent(
    model=llm,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [8]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='69a17573-f2ab-4ed6-9509-041e87d98b55'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "Q4 results of company"}'}, '__gemini_function_call_thought_signatures__': {'call_16860534686268608101': 'Eo8BCowBAWkUfRN/UwWsygg0wgf02PIEoScr+s21joP1I99ZfE/YBBga1iknS/uNY33yo/VOC71iirRNNAWsVmZvlo/WFObIIjLRS/gtvL7C5ioibx2b5w3of6CQlKOCgAJfOVynf6bHoSiscfeh/vVHxu+KqO55yDSIsqFbBDU/uuXEnD9ivwYsSbhwDuypoc4='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f463-92e3-77c6-a0be-49511be757d2-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'Q4 results of company'}, 'id': 'call_16860534686268608101', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m

In [9]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
[{'type': 'text', 'text': 'I have sent an email to team@company.com with the subject "Q4 Results Update". \n\nSince I don\'t have access to your internal company metrics, I used a placeholder in the body. If you\'d like me to include specific numbers, revenue figures, or other details, just let me know and I can send a follow-up or a revised email!', 'extras': {'signature': 'EnsKeQFpFH0Tz81l/XrQzV9ud0/QYAZFzvrZMY9EtUbu4vYWsq0eLwFG/HUmFI+rIhxy2utC+54c74Ueg13czps7whuWjULO7iUb5irUs34VkfIz/AXDZXqiDHrUT7PsOzCd94rEU9Z/Bemsd2XqIvasiMfU7FFlO/8dbAQ='}}]


In [10]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
[{'type': 'text', 'text': 'The operation to delete records from the users table where `active=false` was not executed because it was rejected. Let me know if you need help with anything else.', 'extras': {'signature': 'EnsKeQFpFH0TLR7PS9+rrdr6v1sRji8O/S3VUBTYCJ2a+6nGEVOrxGbENNp39Re1uyok0Hk4oxdYIpHPxnXMoXzq3oPiLb/PLWxrHKh7aK2OwvD1HeAQxMh3YK/EIhllbKer+lCBIm4gynXLbxWrwd8+qNFAGA6ohhT/BEE='}}]


# ⚙️ Section 5: Custom Guardrail — Before-Agent Hook (Input Filter)
Use before_agent() to validate or block requests before any LLM processing begins.

Best for:

Keyword/content filtering
Authentication checks
Rate limiting
Blocking specific categories of requests

In [11]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model=llm,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [12]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
[{'type': 'text', 'text': 'At its core, **machine learning (ML)** is a branch of artificial intelligence (AI) that focuses on building applications that learn from data and improve their accuracy over time **without being explicitly programmed** to do so.\n\nTo understand it, it helps to compare it to traditional computer programming:\n\n*   **Traditional Programming:** You feed a computer **data** and a set of **rules** (code), and the computer spits out **answers**. \n*   **Machine Learning:** You feed a computer **data** and the desired **answers**, and the computer figures out the **rules** (the algorithm) connecting them.\n\n---\n\n### A Simple Analogy\nImagine you want to teach a computer to recognize the difference between an email that is spam and one that is not (ham):\n*   **Traditional way:** You would write thousands of rigid rules (e.g., *"If the email contains the word \'free\' and \'million dollars\', mark it as spam"*). This is hard because spam

# 🔍 Section 6: Custom Guardrail — After-Agent Hook (Output Safety)
Use after_agent() to validate the final agent response before the user sees it.

Best for:

Model-based safety evaluation of outputs
Compliance scanning (e.g. legal, medical, financial disclaimers)
Quality validation
Removing sensitive info that slipped through

In [25]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = models = [
            ChatGoogleGenerativeAI(
                model=model_name,
                google_api_key=google_api_key,
                max_retries=0,
                timeout=30,
            )
            for model_name in free_model_names
        ]
        self.safety_model = models[0].with_fallbacks(self.safety_model[1:])

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        content = result.content
        if isinstance(content, list):
            content = next((b["text"] for b in content if isinstance(b, dict) and b.get("type") == "text"), str(content))

        if "UNSAFE" in content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model=llm,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [26]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
[{'type': 'text', 'text': "I would love to help you with the weather, but I don't know your location! Could you please tell me which city or area you are in?", 'extras': {'signature': 'EnsKeQFpFH0TGXPMiIBOmU0u51epWbmS60jGJvntsC3ISdVJcNlGkjj2n37Gll1vdY7iKbc0AmEVx/8jvOdU/N0Tg20SalzWUzU48M4EJON5EbwyfJDHsVPU71n7axPM1kr/3o4jxJqiK88Vw/FzGruQwrcrZIzeZpehGVE='}}]


# 🧱 Section 7: Layered / Combined Guardrails
Stack multiple guardrails in the middleware=[] array. They execute in order, building layered protection.

User Input

    ↓

[Layer 1] ContentFilterMiddleware    ← Deterministic input filter

    ↓

[Layer 2] PIIMiddleware (input)      ← PII redaction on input

    ↓

[Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools

    ↓

[Layer 4] PIIMiddleware (output)     ← PII redaction on output

    ↓

[Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety

    ↓

User Response

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full layered guardrail stack
production_agent = create_agent(
    model=llm,
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input
       
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!


In [32]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage

# --- Healthcare-specific content filter ---
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        content = last_message.content

        # Gemini returns content as a list of blocks; extract text for the check
        if isinstance(content, list):
            text_content = next(
                (b["text"] for b in content if isinstance(b, dict) and b.get("type") == "text"),
                ""
            )
        else:
            text_content = content

        # Add disclaimer if not already present
        if "medical advice" not in text_content.lower():
            if isinstance(content, list):
                for block in content:
                    if isinstance(block, dict) and block.get("type") == "text":
                        block["text"] += self.DISCLAIMER
                        break
            else:
                last_message.content += self.DISCLAIMER

        return None


# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."


# --- Build the healthcare chatbot ---
healthcare_bot = create_agent(
    model=llm,
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

🏥 Healthcare chatbot with full guardrail stack created!


In [33]:

# Test 1: Safe medical query
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

result

{'messages': [HumanMessage(content='What are symptoms of Type 2 Diabetes?', additional_kwargs={}, response_metadata={}, id='a137018b-9e38-49a2-ab7d-32989d2d03db'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_symptoms', 'arguments': '{"symptoms": "symptoms of Type 2 Diabetes"}'}, '__gemini_function_call_thought_signatures__': {'call_7854274971598767254': 'Eo8BCowBAWkUfRMjX00KU5mN0EpXD5yQu7M2omNCm072d7A0rrSp6/6cm9+pGzdwRTdGFoOMJkgVPiLT3m257+9tFqzSbHKX/DtWdcybQMxcMhcyNBpwfQZHL2ms+tDtAK4Tw1+RlSClLApUaB6jybbfB0zaH/3Bxf+KYtZ4A5eKmCqWCVrNcd9MRyK1rzLS/+8='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f4cf-ae94-7176-89f0-614a196e890a-0', tool_calls=[{'name': 'search_symptoms', 'args': {'symptoms': 'symptoms of Type 2 Diabetes'}, 'id': 'call_7854274971598767254', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 209, 'outpu